In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

In [0]:
catalog_name = 'ecommerce'
schema_name = 'bronze'

df = spark.table(f'{catalog_name}.{schema_name}.bronze_order_items')

In [0]:
display(df.limit(10))

In [0]:
df.select('quantity').distinct().show()

In [0]:
quantity_discrepancies = {
  'Two': '2'
}

df = df.replace(quantity_discrepancies, subset=['quantity'])
df = df.withColumn('quantity', F.col('quantity').cast(IntegerType()))
display(df.select('quantity').distinct())

In [0]:
df = df.withColumn('unit_price', F.regexp_replace('unit_price','[^0-9.,]',''))\
    .withColumn('unit_price', F.regexp_replace('unit_price',',','.'))\
    .withColumn('unit_price', F.col('unit_price').cast(FloatType()))

display(df.select('unit_price').filter(F.col('unit_price').rlike(r'\$')))

In [0]:
df = df.withColumn('discount_pct', F.regexp_replace('discount_pct','%',''))\
    .withColumn('discount_pct', F.col('discount_pct').cast(FloatType()))\
    .withColumn('discount_pct', F.col('discount_pct')*0.01)

display(df.select('discount_pct').limit(5))

In [0]:
channel_description = {
    'web': 'Website',
    'app': 'Mobile App'
}

df = df.replace(channel_description, subset=['channel'])
display(df.select('channel').distinct())

In [0]:
df = df.withColumn('tax_amount', F.regexp_replace('tax_amount','[^0-9.,]',''))\
    .withColumn('tax_amount', F.regexp_replace('tax_amount',',','.'))\
    .withColumn('tax_amount', F.col('tax_amount').cast(FloatType()))

In [0]:
df.printSchema()

In [0]:
df = df.withColumn('dt', F.to_date('dt', 'yyyy-MM-dd'))

df = df.withColumn('order_ts', F.to_timestamp('order_ts', 'yyyy-MM-dd HH:mm:ss'))

df = df.withColumn('processed_time', F.current_timestamp())


In [0]:
df.write.mode('overwrite')\
    .format('delta')\
    .option('mergeSchema', 'true')\
    .saveAsTable(f'{catalog_name}.silver.silver_order_items')